# Stage 1 - Scraping YouTube comments

Goal here is just to pull the raw data we need for the rest of the project.
We're a pair so target is ~20k comments total (10k each).

Builds on the sample `YouTube_Comments_Advanced` notebook from Moodle, but the sample only grabbed
top level comments from a fixed list of videos. To actually reach 20k reliably we:
- search the topic to find videos automatically (so we don't run out of comments)
- also pull the replies, not just top level
- dedup and stop once we hit the target

Output: `../data/raw_comments.csv`

In [4]:
import googleapiclient.discovery
import googleapiclient.errors
import pandas as pd
import time

## Config

The API key is read from `final/.env` - copy `example.env` to `.env` and paste your key in.
Get one from console.cloud.google.com -> enable "YouTube Data API v3" -> create API key.
Each of us used our own key so we don't blow the daily quota (10,000 units/day).

In [14]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")
API_KEY = os.getenv("YOUTUBE_API_KEY")
assert API_KEY, "no key found - copy example.env to .env and add your YouTube API key"

# set this to 0 (your run) or 1 (partner's run)
# each person gets a different slice of video IDs so there's no overlap
SCRAPER_ID = 1   # 0 = even-indexed videos, 1 = odd-indexed videos

SEARCH_QUERY    = "iphone 16 review"
SEED_VIDEO_IDS  = []
TARGET_COMMENTS = 10000   # 10k each -> 20k combined after merge
GET_REPLIES     = True

# output file for this run - merge both in notebook 2
OUT_FILE = f"../data/raw_comments_{SCRAPER_ID}.csv"


In [15]:
api_service_name = "youtube"
api_version = "v3"

youtube = googleapiclient.discovery.build(
    api_service_name, api_version, developerKey=API_KEY)

## 1. Find videos for our topic

`search.list` costs 100 quota units per call (expensive) but each call gives up to 50 video ids,
so a couple of calls is plenty. We sort by relevance, could also do viewCount if we want busier videos.

In [16]:
def search_videos(query, max_videos=60):
    ids = []
    next_token = None
    while len(ids) < max_videos:
        req = youtube.search().list(
            part="id",
            q=query,
            type="video",
            maxResults=50,
            order="relevance",
            pageToken=next_token
        )
        resp = req.execute()
        for item in resp["items"]:
            ids.append(item["id"]["videoId"])
        next_token = resp.get("nextPageToken")
        if not next_token:
            break
    return ids

## 2. Pull comments for one video

Same idea as the sample notebook (loop over pages with nextPageToken) but we also keep the
comment id + parent id so later we know what's a reply, and we read the inline replies that
`part="snippet,replies"` gives us for free (no extra quota).

In [17]:
def get_comments(video_id, with_replies=True):
    rows = []
    request = youtube.commentThreads().list(
        part="snippet,replies",
        videoId=video_id,
        maxResults=100,
        textFormat="plainText"
    )

    while request is not None:
        try:
            response = request.execute()
        except googleapiclient.errors.HttpError as e:
            # comments disabled, video deleted, etc - just skip it
            print("  skipping", video_id, "-", e.resp.status)
            break

        for item in response["items"]:
            top = item["snippet"]["topLevelComment"]
            c = top["snippet"]
            rows.append([
                top["id"],
                None,
                c["authorDisplayName"],
                c["publishedAt"],
                c["likeCount"],
                c["textOriginal"],
                video_id,
                False
            ])

            if with_replies and "replies" in item:
                for r in item["replies"]["comments"]:
                    rc = r["snippet"]
                    rows.append([
                        r["id"],
                        top["id"],
                        rc["authorDisplayName"],
                        rc["publishedAt"],
                        rc["likeCount"],
                        rc["textOriginal"],
                        video_id,
                        True
                    ])

        request = youtube.commentThreads().list_next(request, response)

    return rows

## 3. Main loop - keep scraping until we hit ~20k

Start from the seed videos, then go through the search results, stop once we have enough.
This cell is the slow one (depends on how chatty the videos are).

In [18]:
video_ids = list(SEED_VIDEO_IDS)
video_ids += search_videos(SEARCH_QUERY, max_videos=120)

# deduplicate but keep order
seen = set()
video_ids = [v for v in video_ids if not (v in seen or seen.add(v))]

# split by SCRAPER_ID so each person scrapes different videos - no quota wasted on overlaps
video_ids = video_ids[SCRAPER_ID::2]   # 0 -> even indices, 1 -> odd indices
print(f"scraper {SCRAPER_ID}: {len(video_ids)} videos to process")


scraper 1: 69 videos to process


In [19]:
all_rows = []
seen_ids = set()

for n, vid in enumerate(video_ids, 1):
    if len(all_rows) >= TARGET_COMMENTS:
        break
    print(f"[{n}/{len(video_ids)}] {vid} ... total so far: {len(all_rows)}")
    for row in get_comments(vid, with_replies=GET_REPLIES):
        cid = row[0]
        if cid in seen_ids:
            continue
        seen_ids.add(cid)
        all_rows.append(row)
    time.sleep(0.2)   # be a little polite

print("done, collected", len(all_rows), "comments")

[1/69] dgfmMHuwiD8 ... total so far: 0
[2/69] IMLNj29pJKg ... total so far: 482
[3/69] CkxY54ieosE ... total so far: 1213
[4/69] xGHU9Cp5r04 ... total so far: 1321
[5/69] 6cMJ0I4ub0U ... total so far: 1557
[6/69] f4tabHmCzmI ... total so far: 2419
[7/69] FF9S9NnnH3s ... total so far: 2851
[8/69] vdhH8nUWPn4 ... total so far: 3820
[9/69] tLq2SFv2Lwg ... total so far: 4604
[10/69] WbL7c3bImtI ... total so far: 4891
[11/69] 1ciF-cbNrLw ... total so far: 5157
[12/69] BvA-TAtIJ_k ... total so far: 6695
[13/69] tIVnJtnqANI ... total so far: 6729
[14/69] KMBfrOuRRoQ ... total so far: 7841
[15/69] 7yaDCAoEkGM ... total so far: 7866
[16/69] va3LmAeulRo ... total so far: 8006
[17/69] PbElTZOvMw0 ... total so far: 9285
[18/69] MM-rMtCgwSg ... total so far: 9564
[19/69] FANEYwR-6R0 ... total so far: 9799
[20/69] IG7FrgImhDs ... total so far: 9830
done, collected 13133 comments


In [20]:
df = pd.DataFrame(all_rows, columns=[
    "comment_id", "parent_id", "author", "published_at",
    "like_count", "text", "video_id", "is_reply"
])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13133 entries, 0 to 13132
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   comment_id    13133 non-null  str  
 1   parent_id     2076 non-null   str  
 2   author        13133 non-null  str  
 3   published_at  13133 non-null  str  
 4   like_count    13133 non-null  int64
 5   text          13133 non-null  str  
 6   video_id      13133 non-null  str  
 7   is_reply      13133 non-null  bool 
dtypes: bool(1), int64(1), str(6)
memory usage: 731.2 KB


In [21]:
# quick sanity checks
print("total:", len(df))
print("unique videos:", df['video_id'].nunique())
print("replies vs top level:")
print(df['is_reply'].value_counts())
df.head()

total: 13133
unique videos: 20
replies vs top level:
is_reply
False    11057
True      2076
Name: count, dtype: int64


,comment_id,parent_id,author,published_at,like_count,text,video_id,is_reply
0,Ugz6CFeBDbkw2DcVAZV4AaABAg,NaN,@yuko417,2026-05-21T17:22:15Z,0,Sana inintay ko to 😢 yung iphone 15 na nabili ...,dgfmMHuwiD8,False
1,UgzMlfl_URn9fB8G4TJ4AaABAg,NaN,@emjay_02,2026-05-17T03:50:44Z,0,"Thanks @MaryBautista for this honest review, f...",dgfmMHuwiD8,False
2,UgwwKEeOio5T3VnuUph4AaABAg,NaN,@Eriya0408,2026-04-14T05:00:10Z,0,Okay ba to for EXhorizon?,dgfmMHuwiD8,False
3,UgzGLBRZHyzS-GOcNkB4AaABAg,NaN,@NeciaMagbojos,2026-03-22T16:18:01Z,0,First time apple user and I love my 16 teal. B...,dgfmMHuwiD8,False
4,Ugzg6mDi6X8rOEPP5oN4AaABAg,NaN,@nicholecabradilla1547,2026-03-14T20:20:03Z,0,"miss mary, so i have vivo s30 pro mini now, an...",dgfmMHuwiD8,False


In [22]:
df.to_csv(OUT_FILE, index=False)
print(f"saved {len(df)} rows -> {OUT_FILE}")
print("hand this file to your partner, then run the merge cell in notebook 2")


saved 13133 rows -> ../data/raw_comments_1.csv
hand this file to your partner, then run the merge cell in notebook 2
